In [93]:
import pandas as pd
import numpy as np
import re

In [2]:
error_mapping = {
    #abdominal compositation
    "10P liver PDFF mean error indicator": ["10P Liver PDFF (proton density fat fraction)"],
    "ASAT error indicator": ["Abdominal subcutaneous adipose tissue volume (ASAT)"],
    "Anterior thigh error indicator (left)": ["Anterior thigh muscle fat infiltration (MFI) (left)", "Anterior thigh fat-free muscle volume (left)"],
    "Anterior thigh error indicator (right)": ["Anterior thigh muscle fat infiltration (MFI) (right)", "Anterior thigh fat-free muscle volume (right)"],
    "FR liver PDFF mean error indicator": ["FR liver PDFF mean"],
    "Posterior thigh error indicator (left)": ["Posterior thigh muscle fat infiltration (MFI) (left)", "Posterior thigh fat-free muscle volume (left)"],
    "Posterior thigh error indicator (right)": ["Posterior thigh muscle fat infiltration (MFI) (right)", "Posterior thigh fat-free muscle volume (right)"],
    "VAT error indicator": ["Visceral adipose tissue volume (VAT)"]
}

In [149]:
df = pd.read_table("/project/ukbblatent/clinicaldata/abdominalMRI/Abdominal_composition_82779_MD_01_08_2024_16_54_05.tsv").set_index('f.eid')

In [152]:
df.dropna(how='all').shape

(56224, 31)

In [151]:
all_cols_error = []
for (col_error, cols_item) in error_mapping.items():
    all_cols_error += list(df.filter(regex=f"^{re.escape(col_error.replace(' ', '_'))}*").columns)
    for col_item in cols_item:
        for i in range(4):
                error_cols = df.filter(regex=f"^{re.escape(col_error.replace(' ', '_'))}\.{i}\.\d+$").columns
                item_cols = df.filter(regex=f"^{re.escape(col_item.replace(' ', '_'))}\.{i}\.\d+$").columns
                if len(error_cols) > 0 and len(item_cols) > 0:
                    error_condition = df[error_cols].notna().any(axis=1)
                    df.loc[error_condition, item_cols] = np.nan
df = df.drop(columns=all_cols_error)

In [143]:
76-44

32

In [124]:
col_error

'VAT error indicator'

In [121]:
i=0
df[df.filter(regex=f"^{re.escape(col_error.replace(' ', '_'))}\.{i}\.\d+$").columns].notna()

""
0
1
2
3
4
...
502296
502297
502298
502299


In [95]:
f"^{re.escape(col_item.replace(' ', '_'))}\.{i}\.\d+$"a

'^Visceral_adipose_tissue_volume_\\(VAT\\)\\.3\\.\\d+$'

In [48]:
df_temp = df.melt(id_vars='f.eid', var_name='measurement', value_name='value')
df_temp['instanceID'] = df_temp['measurement'].str.extract(r'\.(\d+\.\d+)$') #let's assume there is no subtype of the instance
df_temp['measurement'] = df_temp['measurement'].str.replace(r'\.\d+\.\d+$', '', regex=True)
df_temp = df_temp.dropna().pivot_table(index=['f.eid', 'instanceID'], 
                    columns='measurement', 
                    values='value')
df_temp.columns.name = None

In [49]:
#show number of not null values for each column
print(df_temp.notnull().sum().sort_values())
print(df_temp.shape)

VAT_error_indicator                                        349
10P_liver_PDFF_mean_error_indicator                        356
FR_liver_PDFF_mean_error_indicator                         423
ASAT_error_indicator                                      1499
Anterior_thigh_error_indicator_(right)                    3195
Posterior_thigh_error_indicator_(right)                   3356
Posterior_thigh_error_indicator_(left)                    4111
Anterior_thigh_error_indicator_(left)                     4124
Total_lean_tissue_volume                                  8521
Total_adipose_tissue_volume                               8521
10P_Liver_PDFF_(proton_density_fat_fraction)              9890
Muscle_fat_infiltration                                  39363
Abdominal_fat_ratio                                      39395
Weight-to-muscle_ratio                                   39431
Total_thigh_fat-free_muscle_volume                       39432
Total_trunk_fat_volume                                 

In [50]:
for (col_error, cols_item) in error_mapping.items():
    for col_item in cols_item:
        df_temp.loc[df_temp[col_error.replace(" ", "_")].notna(), col_item.replace(" ", "_")] = np.nan

In [51]:
df_temp = df_temp.drop(columns=list([k.replace(" ", "_") for k in error_mapping.keys()]))

In [52]:
#show number of not null values for each column
print(df_temp.notnull().sum().sort_values())
print(df_temp.shape)

Total_lean_tissue_volume                                  8521
Total_adipose_tissue_volume                               8521
10P_Liver_PDFF_(proton_density_fat_fraction)              9890
Muscle_fat_infiltration                                  39363
Abdominal_fat_ratio                                      39395
Weight-to-muscle_ratio                                   39431
Total_thigh_fat-free_muscle_volume                       39432
Total_abdominal_adipose_tissue_index                     40080
Total_trunk_fat_volume                                   40080
FR_liver_PDFF_mean                                       50788
Posterior_thigh_muscle_fat_infiltration_(MFI)_(left)     55937
Anterior_thigh_muscle_fat_infiltration_(MFI)_(left)      55975
Posterior_thigh_fat-free_muscle_volume_(left)            56013
Anterior_thigh_fat-free_muscle_volume_(left)             56052
Posterior_thigh_muscle_fat_infiltration_(MFI)_(right)    56651
Posterior_thigh_fat-free_muscle_volume_(right)         

In [55]:
df_temp = df_temp.dropna(how='all')

In [58]:
df_temp.reset_index().instanceID.unique()

array(['2.0', '3.0'], dtype=object)